In [1]:
from sympleq.core.circuits.gates import GATES
from sympleq.core.symmetries.conditional_hamiltonian import ConditionalHamiltonian
from sympleq.models.random_hamiltonian import random_gate_symmetric_hamiltonian
from sympleq.core.symmetries.pauli import pauli_reduce
from sympleq.core.symmetries.clifford import min_qudit_clifford_symmetry
from sympleq.core.circuits.gate_decomposition_to_circuit import gate_to_circuit
import numpy as np
from numpy.random import default_rng

In [2]:
# Example SWAP Symmetry
def generate_symmetry(n_qudits, n_paulis):
    P_sym = random_gate_symmetric_hamiltonian(GATES.H, dimension=2, qudit_indices=tuple([0]),
                                                n_paulis=n_paulis, n_qudits=n_qudits)
    h_red, conditioned_hamiltonians, C_F, all_phases = pauli_reduce(P_sym)
    F, Sy, T = min_qudit_clifford_symmetry(h_red)
    C_F = gate_to_circuit(F, dimensions=[2 for i in range(n_qudits)])
    symmetrised = T.inverse().act(h_red, tuple(np.arange(n_qudits)))
    assert C_F.act(h_red).is_close(h_red, literal=False)
    assert Sy.act(symmetrised, tuple(np.arange(n_qudits))).is_close(symmetrised, literal=False)
    return h_red, Sy, T

In [3]:
Hamiltonian, symmetry, transformer = generate_symmetry(3, 16)

conditional_hamiltonian = ConditionalHamiltonian(Hamiltonian, symmetry, transformer)

In [4]:
# get a specific conditional hamiltonian
print(conditional_hamiltonian.block_variables['conditional_dimensions']) # should be [2, 1, 1]
# Conditional dimensions tell us, how many unique eigenvalues, i.e. options, there are for each block
# for example, the first block has 2 eigenvalues

# select_hamiltonian allows to specify which eigenvalues are selected
# [0,0,0] corresponds to the first eigenvalue of the first block, second block and third block
# [1,0,0] corresponds to the second eigenvalue of the first block and the first eigenvalue of the second and third block
print(conditional_hamiltonian.select_hamiltonian([0,0,0]))

[2, 1, 1]
(-2.8284271247461907+3.003954790883355e-16j)  |x0z0 x0z0 | 0 
(1.0000000000000004+0j)                       |x0z0 x0z1 | 0 
(-0.41421356237309503+1.5019773954416774e-16j)|x0z1 x0z0 | 0 
(1.0000000000000004+0j)                       |x0z1 x0z1 | 0 
(1.0000000000000004+0j)                       |x0z0 x1z0 | 0 
(-0.41421356237309503+1.5019773954416774e-16j)|x0z1 x1z0 | 0 
(1.0000000000000004+0j)                       |x1z0 x0z0 | 0 
(-1.4142135623730954+1.5019773954416774e-16j) |x1z0 x1z0 | 0 



In [5]:
# testing that the conditional hamiltonians together make the original hamiltonian
conditional_hamiltonian.test_conditional_hamiltonian()

True

In [6]:
# iterate through all conditional hamiltonians
for c in conditional_hamiltonian:
    print(c)

(-2.8284271247461907+3.003954790883355e-16j)  |x0z0 x0z0 | 0 
(1.0000000000000004+0j)                       |x0z0 x0z1 | 0 
(-0.41421356237309503+1.5019773954416774e-16j)|x0z1 x0z0 | 0 
(1.0000000000000004+0j)                       |x0z1 x0z1 | 0 
(1.0000000000000004+0j)                       |x0z0 x1z0 | 0 
(-0.41421356237309503+1.5019773954416774e-16j)|x0z1 x1z0 | 0 
(1.0000000000000004+0j)                       |x1z0 x0z0 | 0 
(-1.4142135623730954+1.5019773954416774e-16j) |x1z0 x1z0 | 0 

(2.82842712474619-3.923693659000591e-16j)   |x0z0 x0z0 | 0 
(0.9999999999999997+0j)                     |x0z0 x0z1 | 0 
(2.4142135623730945-1.9618468295002955e-16j)|x0z1 x0z0 | 0 
(0.9999999999999997+0j)                     |x0z1 x0z1 | 0 
(0.9999999999999997+0j)                     |x0z0 x1z0 | 0 
(2.4142135623730945-1.9618468295002955e-16j)|x0z1 x1z0 | 0 
(0.9999999999999997+0j)                     |x1z0 x0z0 | 0 
(1.414213562373095-1.9618468295002955e-16j) |x1z0 x1z0 | 0 

